In [ ]:
import os
import joblib
import numpy as np
from google.colab import drive

# =============================================================================
# 1. KẾT NỐI DRIVE & ĐỊNH NGHĨA ĐƯỜNG DẪN
# =============================================================================
try:
    drive.mount('/content/drive')
except:
    pass

# Đường dẫn gốc (Anh kiểm tra lại nếu khác)
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
DATA_PATH = os.path.join(BASE_DIR, "Multi_Data/")      # Nơi chứa dữ liệu đa lớp
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "Models-CNN/") # Nơi lưu model CNN

# Tạo thư mục lưu model nếu chưa có
if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

print("-" * 60)
print(f"📂 Dữ liệu lấy từ: {DATA_PATH}")
print(f"📂 Model sẽ lưu tại: {MODEL_SAVE_PATH}")

# =============================================================================
# 2. LOAD DỮ LIỆU ĐA LỚP
# =============================================================================
print("-" * 60)
print("⏳ Đang load dữ liệu (X_train, y_train...)...")

try:
    # Load dữ liệu đã chuẩn hóa và One-Hot
    X_train = joblib.load(DATA_PATH + 'X_train_multi.pkl')
    y_train = joblib.load(DATA_PATH + 'y_train_multi.pkl')
    X_test = joblib.load(DATA_PATH + 'X_test_multi.pkl')
    y_test = joblib.load(DATA_PATH + 'y_test_multi.pkl')

    # Load trọng số lớp
    class_weights = joblib.load(DATA_PATH + 'class_weights_multi.pkl')

    print(f"✅ Load xong! Kích thước gốc X_train: {X_train.shape}")

    # =========================================================================
    # 3. RESHAPE DỮ LIỆU CHO CNN (QUAN TRỌNG NHẤT)
    # =========================================================================
    # CNN 1D yêu cầu đầu vào 3 chiều: (Số mẫu, Số đặc trưng, 1 kênh)
    print("🔄 Đang Reshape dữ liệu sang 3D cho CNN...")

    n_features = X_train.shape[1]

    X_train_cnn = X_train.reshape(X_train.shape[0], n_features, 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], n_features, 1)

    print(f"✅ Kích thước mới cho CNN (X_train_cnn): {X_train_cnn.shape}")
    print(f"✅ Số lớp đầu ra (y_train): {y_train.shape[1]} (Phải là 5)")

except FileNotFoundError as e:
    print(f"❌ LỖI: Không tìm thấy file dữ liệu. Vui lòng kiểm tra đường dẫn!\n{e}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

# =============================================================================
# 1. XÂY DỰNG MÔ HÌNH 1D-CNN (CHO 5 LỚP)
# =============================================================================
def build_multiclass_cnn(input_shape, n_classes):
    model = Sequential(name="CNN_Multiclass_NIDS")

    # --- BLOCK 1: Trích xuất đặc trưng cơ bản ---
    model.add(Conv1D(filters=32, kernel_size=3, padding='same', activation='relu', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # --- BLOCK 2: Trích xuất đặc trưng sâu hơn ---
    model.add(Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # --- FLATTEN: Duỗi thẳng dữ liệu ---
    model.add(Flatten())

    # --- CLASSIFIER: Phân loại ---
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5)) # Giữ Dropout 0.5 để chống Overfitting tốt

    # --- OUTPUT LAYER (QUAN TRỌNG) ---
    # 5 Nơ-ron (cho 5 lớp) + Softmax (xác suất đa lớp)
    model.add(Dense(n_classes, activation='softmax'))

    # Compile
    optimizer = Adam(learning_rate=0.001)
    model.compile(loss='categorical_crossentropy', # Bắt buộc cho đa lớp One-Hot
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model

# Khởi tạo mô hình
if 'X_train_cnn' in locals():
    input_shape = (X_train_cnn.shape[1], 1) # (37, 1)
    n_classes = y_train.shape[1]            # 5

    model_cnn = build_multiclass_cnn(input_shape, n_classes)
    model_cnn.summary()

    # =========================================================================
    # 2. CẤU HÌNH CALLBACKS & HUẤN LUYỆN
    # =========================================================================
    print("-" * 60)
    print("🚀 BẮT ĐẦU HUẤN LUYỆN...")

    checkpoint_path = os.path.join(MODEL_SAVE_PATH, 'Scenario2_CNN_Multiclass_Best.keras')

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1),
        EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]

    history = model_cnn.fit(
        X_train_cnn, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=256,
        class_weight=class_weights, # Cân bằng dữ liệu
        callbacks=callbacks,
        verbose=1
    )

    print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

    # =========================================================================
    # 3. VẼ BIỂU ĐỒ ĐÁNH GIÁ
    # =========================================================================
    plt.figure(figsize=(14, 6))

    # Biểu đồ Loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss', color='blue')
    plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
    plt.title('Hàm mất mát (Loss)')
    plt.xlabel('Vòng (Epochs)'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True)

    # Biểu đồ Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Acc', color='green')
    plt.plot(history.history['val_accuracy'], label='Val Acc', color='red')
    plt.title('Độ chính xác (Accuracy)')
    plt.xlabel('Vòng (Epochs)'); plt.ylabel('Accuracy')
    plt.legend(); plt.grid(True)
    plt.show()

else:
    print("❌ LỖI: Không tìm thấy dữ liệu X_train_cnn. Hãy chạy lại BƯỚC 1 trước!")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time

# =============================================================================
# BƯỚC 3: ĐÁNH GIÁ MÔ HÌNH 1D-CNN (ĐỂ SO SÁNH)
# =============================================================================
if 'model_cnn' in locals() and 'X_test_cnn' in locals():
    print("-" * 60)
    print("🧐 ĐANG CHẤM ĐIỂM MÔ HÌNH TRÊN TẬP TEST...")

    # 1. Dự đoán & Đo tốc độ
    start_time = time.time()
    y_pred_probs = model_cnn.predict(X_test_cnn, verbose=0) # Tắt verbose cho gọn
    end_time = time.time()

    # Tính thời gian trung bình mỗi mẫu (rất quan trọng để so sánh hiệu năng)
    inference_time = (end_time - start_time) / len(X_test_cnn) * 1_000_000 # đổi ra microseconds
    print(f"⏱️ Tốc độ xử lý trung bình: {inference_time:.2f} µs/mẫu")

    # 2. Chuyển đổi kết quả (Xác suất -> Nhãn số)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Chuyển y_test (One-Hot) về dạng số nguyên (nếu chưa chuyển)
    if y_test.ndim > 1 and y_test.shape[1] > 1:
        y_true = np.argmax(y_test, axis=1)
    else:
        y_true = y_test

    # 3. Tên các lớp (Theo thứ tự mapping)
    class_names = ['Benign', 'DoS/DDoS', 'PortScan', 'BruteForce', 'Other']

    # --- A. Báo cáo chi tiết ---
    print("\n📊 CLASSIFICATION REPORT (1D-CNN):")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # --- B. Vẽ Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    # Vẽ heatmap với số lượng mẫu (fmt='d')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.ylabel('Thực tế (Actual)')
    plt.xlabel('Dự đoán (Predicted)')
    plt.title('Confusion Matrix - 1D CNN Multiclass')
    plt.show()

    # --- C. Phân tích nhanh ---
    # Tính độ chính xác từng lớp để anh dễ so sánh
    # (Đường chéo chính chia cho tổng hàng)
    class_accuracy = cm.diagonal() / cm.sum(axis=1)
    print("\n🔍 ĐỘ CHÍNH XÁC TỪNG LỚP (RECALL):")
    for i, acc in enumerate(class_accuracy):
        print(f"   - {class_names[i]}: {acc*100:.2f}%")

else:
    print("❌ LỖI: Thiếu biến 'model_cnn' hoặc 'X_test_cnn'. Hãy chạy Bước 1 & 2 trước.")

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# 1. Dự đoán xác suất (Lấy kết quả dạng số thực từ 0.0 đến 1.0)
# Lưu ý: Dùng X_test_cnn (dữ liệu đã reshape 3D)
print("⏳ Đang tính toán xác suất dự đoán...")
# y_pred_prob = model_cnn.predict(X_test_cnn).ravel() # This was causing the error
y_pred_probs = model_cnn.predict(X_test_cnn)

# 2. Tính toán FPR, TPR và ngưỡng (Thresholds) cho từng lớp

# Chuyển y_test (One-Hot) về dạng số nguyên (nếu chưa chuyển)
if y_test.ndim > 1 and y_test.shape[1] > 1:
    y_true_labels = np.argmax(y_test, axis=1)
else:
    y_true_labels = y_test

n_classes = y_test.shape[1]
class_names = ['Benign', 'DoS/DDoS', 'PortScan', 'BruteForce', 'Other'] # Make sure this matches your actual class names

plt.figure(figsize=(12, 10))

# Compute ROC curve and ROC area for each class
for i in range(n_classes):
    # Get true binary labels for the current class (one-vs-rest)
    y_true_binary = (y_true_labels == i).astype(int)

    # Get predicted probabilities for the current class
    y_score_class = y_pred_probs[:, i]

    fpr, tpr, _ = roc_curve(y_true_binary, y_score_class)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, lw=2, label=f'ROC curve of class {class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Random guess line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Tỷ lệ Báo động giả)')
plt.ylabel('True Positive Rate (Tỷ lệ Phát hiện đúng)')
plt.title('Receiver Operating Characteristic (ROC) - 1D CNN Multiclass (One-vs-Rest)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()